In [1]:
import torch
import triton
import triton.language as tl
from triton.testing import do_bench

print(torch.__version__, triton.__version__, torch.cuda.get_device_name())

torch.backends.cuda.matmul.allow_tf32 = True

2.14.1+cu130 3.8.0 NVIDIA GeForce RTX 3060


In [6]:
@triton.autotune(
    configs=[
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "SPLIT_K": 1},
            num_warps=8,
            num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 32, "SPLIT_K": 1},
            num_warps=8,
            num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "SPLIT_K": 2},
            num_warps=8,
            num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 32, "SPLIT_K": 2},
            num_warps=8,
            num_stages=4,
        ),
    ],
    key=["M", "N", "K"],
    reset_to_zero=["c_ptr"],
)
@triton.jit
def matmul_kernel2(
    a_ptr,
    b_ptr,
    c_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    SPLIT_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    pid_k = tl.program_id(2)

    K_CHUNK = tl.cdiv(K, SPLIT_K)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K_CHUNK, BLOCK_K):
        am_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        am_mask = am_offsets < M
        ak_offsets = pid_k * K_CHUNK + k + tl.arange(0, BLOCK_K)
        ak_mask = ak_offsets < min((pid_k + 1) * K_CHUNK, K)

        a = tl.load(
            a_ptr + am_offsets[:, None] * stride_am + ak_offsets * stride_ak,
            mask=am_mask[:, None] & ak_mask,
            other=0.0,
        )

        bn_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        bn_mask = bn_offsets < N
        bk_offsets = pid_k * K_CHUNK + k + tl.arange(0, BLOCK_K)
        bk_mask = bk_offsets < min((pid_k + 1) * K_CHUNK, K)

        b = tl.load(
            b_ptr + bk_offsets[:, None] * stride_bk + bn_offsets * stride_bn,
            mask=bk_mask[:, None] & bn_mask,
            other=0.0,
        )

        acc = tl.dot(a, b, acc=acc)

    cm_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cm_mask = cm_offsets < M
    cn_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    cn_mask = cn_offsets < N

    tl.atomic_add(
        c_ptr + cm_offsets[:, None] * stride_cm + cn_offsets * stride_cn,
        acc,
        mask=cm_mask[:, None] & cn_mask,
    )


def matmul2(a, b, **kwargs):
    assert a.ndim == b.ndim == 2
    assert a.shape[1] == b.shape[0]
    assert a.device == b.device
    assert a.dtype == b.dtype

    M, K = a.shape
    _, N = b.shape

    out = torch.zeros((M, N), device=a.device, dtype=torch.float32)

    grid = lambda meta: (
        triton.cdiv(M, meta["BLOCK_M"]),
        triton.cdiv(N, meta["BLOCK_N"]),
        meta["SPLIT_K"],
    )

    matmul_kernel2[grid](
        a,
        b,
        out,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        out.stride(0),
        out.stride(1),
    )

    return out.to(a.dtype)

In [7]:
torch.testing.assert_close(matmul2(a, b), a @ b, atol=1e-2, rtol=1e-2)
print(K, matmul_kernel2.best_config)  # see which config won

AssertionError: Tensor-likes are not close!

Mismatched elements: 9950 / 1048576 (0.9%)
Greatest absolute difference: 0.265625 at index (842, 338) (up to 0.01 allowed)
Greatest relative difference: 426.5 at index (399, 1005) (up to 0.01 allowed)

In [8]:
torch.backends.cuda.matmul.allow_tf32 = False  # make sure the reference is real fp32
ref = a.float() @ b.float()
ours = matmul2(a, b).float()
theirs = (a @ b).float()

print("K =", a.shape[1], matmul_kernel2.best_config)
print("ours   vs fp32:", (ours - ref).abs().max().item())
print("cuBLAS vs fp32:", (theirs - ref).abs().max().item())
print(
    "fp16 spacing at max |ref|:",
    (torch.finfo(torch.float16).eps * ref.abs().max()).item(),
)

K = 16384 BLOCK_M: 64, BLOCK_N: 128, BLOCK_K: 32, SPLIT_K: 1, num_warps: 8, num_ctas: 1, num_stages: 4, maxnreg: None
ours   vs fp32: 0.2606201171875
cuBLAS vs fp32: 0.6121826171875
fp16 spacing at max |ref|: 0.6244892477989197


In [9]:
rel_err = (ours - ref).norm() / ref.norm()
assert rel_err < 1e-3, (
    rel_err
)  # fp16 output rounding is ~5e-4; a real bug is far larger

In [11]:
a.shape, b.shape

(torch.Size([1024, 16384]), torch.Size([16384, 1024]))

In [14]:
1024 / 64, 1024 / 128, 16384 / 32

(16.0, 8.0, 512.0)

In [5]:
print(f"{'k_size':8s} {'cublas':14s} {'yours':14s}")
for size in [
    16,
    32,
    64,
    128,
    256,
    512,
    1024,
    2048,
    4096,
    8192,
    16384,
]:  # beyond 64 out of shared memory
    a = torch.randn(1024, size, device="cuda", dtype=torch.float16)
    b = torch.randn(size, 1024, device="cuda", dtype=torch.float16)
    cublas = lambda: torch.matmul(a, b)
    yours = lambda: matmul2(a, b, BLOCK_K=size)

    torch.testing.assert_close(cublas(), yours(), atol=1e-1, rtol=1e-1)

    ms_cublas = do_bench(cublas, warmup=10, rep=100)
    ms_triton = do_bench(yours, warmup=10, rep=100)
    print(f"{size:<8d} {ms_cublas:<14.3f} {ms_triton:<14.3f}")

k_size   cublas         yours         
16       0.013          0.094         
32       0.014          0.094         
64       0.016          0.097         
128      0.022          0.101         
256      0.030          0.110         
512      0.061          0.127         
1024     0.113          0.161         
2048     0.184          0.242         
4096     0.354          0.419         
8192     0.695          0.774         


AssertionError: Tensor-likes are not close!

Mismatched elements: 13 / 1048576 (0.0%)
Greatest absolute difference: 0.151123046875 at index (363, 378) (up to 0.1 allowed)
Greatest relative difference: 151.875 at index (479, 15) (up to 0.1 allowed)

In [ ]:
print(f"{'mn_size':8s} {'cublas':14s} {'yours':14s}")
for size in [
    16,
    32,
    64,
    128,
    256,
    512,
    1024,
    2048,
    4096,
    8192,
    16384,
]:  # beyond 64 out of shared memory
    a = torch.randn(size, 1024, device="cuda", dtype=torch.float16)
    b = torch.randn(1024, size, device="cuda", dtype=torch.float16)
    cublas = lambda: torch.matmul(a, b)
    yours = lambda: matmul2(a, b, BLOCK_K=size)

    torch.testing.assert_close(cublas(), yours(), atol=1e-1, rtol=1e-1)

    ms_cublas = do_bench(cublas, warmup=10, rep=100)
    ms_triton = do_bench(yours, warmup=10, rep=100)
    print(f"{size:<8d} {ms_cublas:<14.3f} {ms_triton:<14.3f}")

mn_size  cublas         yours         
16       0.009          0.028         
32       0.008          0.028         
64       0.009          0.033         
128      0.008          0.034         
256      0.015          0.036         
512      0.033          0.066         
1024     0.114          0.163         
2048     0.363          0.564         
4096     1.348          2.160         
8192     5.442          8.665         
16384    22.417         35.406        


In [ ]:
cublas()

tensor([[-53.7188,  11.5703,  51.0625,  ...,  48.7500,  14.7344,  -9.1094],
        [ -5.8828,  27.6562, -15.7969,  ...,  22.8438,  21.1406,  37.5000],
        [ 24.3750,  -0.3867,  14.2734,  ...,  10.3984,  36.8125,  34.3750],
        ...,
        [-29.2344, -33.9062,  -4.5234,  ...,  56.0312,  58.6875,  38.9062],
        [-21.4844,  54.5312,  49.4375,  ...,  40.1562,   9.2891,  18.5938],
        [ -9.9297,  64.0000, -19.5625,  ...,  35.4688, -39.8750,   4.0977]],
       device='cuda:0', dtype=torch.float16)

In [ ]:
yours()

tensor([[-53.7188,  11.5625,  51.0625,  ...,  48.7500,  14.7422,  -9.1094],
        [ -5.8828,  27.6562, -15.7969,  ...,  22.8281,  21.1562,  37.5000],
        [ 24.3750,  -0.3726,  14.2734,  ...,  10.3906,  36.8438,  34.3750],
        ...,
        [-29.2344, -33.9062,  -4.5234,  ...,  56.0312,  58.6875,  38.9062],
        [-21.4844,  54.5312,  49.4375,  ...,  40.1562,   9.2812,  18.5938],
        [ -9.9141,  64.0000, -19.5625,  ...,  35.4688, -39.9062,   4.0977]],
       device='cuda:0', dtype=torch.float16)